# Week 5 Problem Set: The Null Result

**Instructions:** Complete all four tasks.

The field director says canvassing doesn't work. Your job: figure out whether the evidence supports that claim.

---

**Lying with data: the checklist so far**

1. **W1:** Conflating fixed and marginal costs to make a tactic look cheaper than it is.
2. **W2:** Presenting an observational comparison as a causal effect.
3. **W3:** Applying a result from one setting to a different one without argument.
4. **W4:** Cherry-picking the winning arm from a multi-arm test.
5. **W5:** Treating an underpowered null as evidence of no effect.

### Before you start

**Save your own copy first.** Go to **File → Save a copy in Drive**. A new tab opens with your own copy. Work in that tab; edits to the original are not saved.

**The data loads itself.** There is nothing to download or upload. The setup cell below pulls the data straight from the course repository; you just need to be online.

Stuck? See the Colab troubleshooting guide on the syllabus.

## Setup


In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf

In [ ]:
df = pd.read_csv('https://raw.githubusercontent.com/joshuakalla/data_science_campaigns_26/'
                 'main/weeks/wk05_power_and_nulls/data/canvassing_pilot.csv')
df.shape

## Task 1: The pilot results

Compute the ATE from the pilot data and run randomization inference.


In [ ]:
# Compute the ATE
treated_mean = df[df['treatment'] == 1]['turned_out'].mean()
control_mean = df[df['treatment'] == 0]['turned_out'].mean()
observed_ate = treated_mean - control_mean
print(f'Control turnout: {control_mean:.3f}')
print(f'Treatment turnout: {treated_mean:.3f}')
print(f'ATE: {observed_ate:+.4f}')

In [ ]:
# Randomization inference — same .sample(frac=1).values shuffle as W3 and W4.
np.random.seed(42)
fake_ates = []
for i in range(1000):
    shuffled = df['treatment'].sample(frac=1).values  # shuffle labels, keep outcomes fixed
    fake_ate = df['turned_out'][shuffled == 1].mean() - df['turned_out'][shuffled == 0].mean()
    fake_ates.append(fake_ate)

p_value = np.mean([abs(f) >= abs(observed_ate) for f in fake_ates])
print(f'p-value: {p_value:.3f}')

You should get about **0.84**. The field director's report said **0.76** — she ran a regression, we ran randomization inference, and on 400 observations the two methods do not land on the same p-value. You will see her 0.76 come back in Task 1b when you run the regression. Neither is anywhere near 0.05, which is the only thing that matters here.

**Question 1:** Is the result statistically significant? In one sentence, explain what the p-value tells you and what it does *not* tell you.

*Your answer:*


## Task 1b: The confidence interval (what can the pilot rule out?)

A non-significant p-value is hard to interpret on its own. The **confidence interval** is more useful: it is the range of effect sizes *consistent with the data*, the values we cannot rule out. Run the cell to compute it for the pilot.

In [ ]:
ate = observed_ate
se  = np.sqrt(treated_mean*(1-treated_mean)/200 + control_mean*(1-control_mean)/200)
lo, hi = ate - 1.96*se, ate + 1.96*se
print(f'ATE:                     {100*ate:+.1f} pp')
print(f'Standard error:          {100*se:.1f} pp')
print(f'95% confidence interval: [{100*lo:+.1f}, {100*hi:+.1f}] pp')

**Question 1b:** The 95% confidence interval is about **[-8, +11] pp**.
(a) Does it include 0? (b) Does it include +10 pp? (c) In one or two sentences: what does the *width* of this interval tell you about whether the pilot can rule out either "canvassing does nothing" or "canvassing works a lot"?

*Your answer:*

Now read the same result as a **regression table**, the columns Week 2 deferred and Week 3 skipped. Ignore the `Intercept` row, the `t` column, and the diagnostics panels; you want the four columns from the slides. Run:

In [ ]:
smf.ols('turned_out ~ treatment', data=df).fit().summary()

**Question 1c:** Find the `treatment` row. Which number is the ATE, which is the standard error, which two numbers are the confidence interval? And why does `P>|t|` make this "not significant"?

*Your answer:*

## Task 2: Power simulation

Assume the true effect of canvassing is +2 percentage points and control turnout is 42%. Simulate 1,000 fake experiments at the pilot's sample size (n = 400) to compute the power.

For each fake experiment, we generate the data, then check significance using a quick randomization inference (200 shuffles). Power = the fraction of experiments where p < 0.05.


In [ ]:
# Power simulation at n = 400
# What this does: generate 1,000 fake experiments where canvassing truly works (+2 pp)
# and count how many produce p < 0.05

np.random.seed(42)
n_per_group = 200
true_control_rate = 0.42
true_treatment_rate = 0.44  # true effect = +2 pp

detected = 0

for i in range(1000):
    if i % 200 == 0:
        print(f'Experiment {i}/1000...')
    
    # Generate one fake experiment
    control = np.random.binomial(1, true_control_rate, size=n_per_group)
    treatment = np.random.binomial(1, true_treatment_rate, size=n_per_group)
    ate = treatment.mean() - control.mean()
    
    # Quick RI: shuffle 200 times
    all_outcomes = np.concatenate([control, treatment])
    ri_count = 0
    for j in range(200):
        shuffled = np.random.permutation(all_outcomes)
        ri_ate = shuffled[:n_per_group].mean() - shuffled[n_per_group:].mean()
        if abs(ri_ate) >= abs(ate):
            ri_count += 1
    p = ri_count / 200
    
    if p < 0.05:
        detected += 1

power = detected / 1000
print(f'\nPower at n = {n_per_group * 2}: {power:.3f}')

**Before you move on:** In the power simulation above, `true_treatment_rate` is set to 0.44 (control rate 0.42, so the true effect is +2 pp). What would happen to the power number if you changed `true_treatment_rate` to 0.43 (a true effect of +1 pp instead of +2 pp)? Would power go up or down? In one sentence, explain why.

**Your answer:**

*Replace this text with your answer.*

**Question 2:** What is the power? In one sentence, explain what this number means for the field director's conclusion.

*Your answer:*


## Task 2a: Your first simulation from scratch

A canvassing volunteer claims she convinced 15 out of 20 people she talked to (75%). You’re skeptical. Maybe people were just being polite, and the true persuasion rate is 50% (no better than a coin flip).

In the empty cell below, write a simulation to test this. Specifically:

1. Use `np.random.binomial(1, 0.50, size=20)` to simulate 20 conversations where the true success rate is 50%. This gives you an array of 0s and 1s. Use `.sum()` to count how many “successes” you got.
2. Wrap that in a `for` loop that repeats 1,000 times. Store each count in a list.
3. After the loop, compute the fraction of simulations where you got 15 or more successes. Print it.

This is a complete simulation from scratch, not a modification of existing code. You’ve seen every piece (`np.random.binomial`, `for` loops, `.sum()`, `np.mean(...)`) in earlier weeks. Now put them together yourself. Writing one from scratch for the first time is hard. Give yourself time, and test each piece on its own if the whole thing does not run.

*Check: the fraction should be small (under 5%). If you get a number close to 0, that’s correct. Getting 15/20 by chance when the true rate is 50% is very unlikely.*

In [ ]:
np.random.seed(42)

# YOUR CODE HERE
# End with a line assigning your answer to `fraction_significant`.

fraction_significant

**In one sentence:** based on your simulation, is the volunteer’s claim of 15/20 consistent with a true rate of 50%, or is it evidence that she really is persuading people?

*Replace this text with your answer.*

## Task 2b: Power at a larger sample

You just saw that power at n = 400 is under 5% in this run (it lands between about 5% and 7% depending on the seed). In the empty cell below, write the same power simulation but change **one number**: set `n_per_group = 1000` (so the total sample is 2,000 instead of 400). Copy the key lines from the Task 2 simulation above and change that one parameter.

You can use fewer repetitions to keep it fast: 500 experiments with 200 RI shuffles each is enough.

*Check: power should be noticeably higher than 6%, but still well below 80%.*

In [ ]:
# YOUR CODE HERE: power simulation at n_per_group = 1000
# End with a line assigning the power to `power_1000`.

power_1000

**In one sentence:** at n = 2,000, could the field director confidently conclude “canvassing doesn’t work” if she got a null result?

*Replace this text with your answer.*

## Task 3: How large would the experiment need to be?

Compute power at several sample sizes to find the one that gives approximately 80% power.

(This uses a faster approximation for significance checking so it runs in a reasonable time.)


In [ ]:
# Power at different sample sizes
# Using a fast approximation: significant if |ATE| > threshold
# where threshold is calibrated to give the same answer as RI

np.random.seed(42)
true_control_rate = 0.42
true_treatment_rate = 0.44

sample_sizes = [400, 1000, 2000, 5000, 8000, 10000, 15000, 20000]

print(f'{"n (total)":>10}  {"Power":>8}')
print('-' * 22)

for n_total in sample_sizes:
    n_per = n_total // 2
    # Approximate significance threshold
    threshold = 1.96 * np.sqrt(2 * 0.43 * 0.57 / n_per)
    detected = 0
    for i in range(1000):
        control = np.random.binomial(1, true_control_rate, size=n_per)
        treatment = np.random.binomial(1, true_treatment_rate, size=n_per)
        ate = treatment.mean() - control.mean()
        if abs(ate) > threshold:
            detected += 1
    power = detected / 1000
    print(f'{n_total:>10,}  {power:>8.1%}')

**Question 3a:** Approximately what sample size gives 80% power to detect a +2 pp effect?

*Your answer:*

**Question 3b:** Using the per-door cost from the cell below, how much would the treatment group of an experiment that size cost? Is that a realistic budget for a single campaign?

*Your answer:*


## Task 3b: Cost of the experiment

In the empty cell below, write code that computes and prints the cost of the treatment group for an experiment with 80% power. Use the sample size you identified in Task 3a.

**Watch the units.** A canvass conversation costs about \$20, but you are not buying conversations, you are buying **doors knocked**, and only about 30% of them open. So a door costs \$20 × 0.30 = **\$6**, and the cost is `n_treated * 6`. Multiplying the household count by \$20 would price every door as though someone answered it.

There is no trick here beyond that. It's multiplication. The point is to turn the sample-size number into a dollar amount so you can use it in the memo.

In [ ]:
# YOUR CODE HERE
# End with a line assigning the total cost to `study_cost`.

study_cost

## Task 4: Design the experiment that could answer the question

The pilot couldn't tell whether canvassing works. It was too small. Your job now is to **design the experiment that could actually settle it**, and to commit in advance to what you'll do with the result. This is not another memo arguing about the pilot.

Write a **one-page design brief** (about 300–400 words; headings are fine, and this is not a 250-word memo). It must include all four:

**(a) The decision rule, pre-committed.** Before you see any data, state what you will recommend for each outcome. For example: "If the experiment estimates an effect of at least +___ pp with a confidence interval excluding zero, I recommend scaling canvassing. If it can rule out effects above +___ pp, I recommend reallocating to digital. If it comes back underpowered or inconclusive, I recommend ___." Pre-committing is what stops you from cherry-picking the story after the fact (the Week 4 lesson).

**(b) The design.** The unit you randomize, treatment vs. control, the **outcome** you'll measure and how (voter-file turnout or a survey: which, and why), and the **one primary comparison** you are pre-registering.

**(c) Sample size and cost.** Use your Task 3 result: the sample size that gives about 80% power to detect a +2 pp effect, and what the treatment group costs at \$6 a door knocked (Task 3b). Justify the number by the power target rather than just asserting it.

**(d) Steelman: "don't run it."** Construct the strongest version of the argument for *skipping* the experiment and deciding now: cost, time, the fact that the published literature already exists. Then say whether you accept it.

*This brief is a miniature of the powered, pre-registered design your final project will ask for in its appendix. Keep it.*

*Your design brief:*

---

**Due at 4:00pm on Wednesday Oct 14, after Exam 1**, to the **problem set** assignment on Canvas. Whatever you had at 6:00pm in class already went to the separate **in-class** assignment; that one is your attendance credit and you do not resubmit it.


## Before you submit

1. **Runtime → Restart session and run all.** Do this *after* you have finished every task and written your memo. It clears every variable and runs the notebook from top to bottom, in order, so the version you hand in is one that actually works start to finish.
2. **Check that every cell actually ran.** Scroll from the top. Every code cell should show a number in its left margin and its output below it. If the run stopped at a cell with an error, that is a cell you have not finished. Fix it, then restart and run all again.
3. **File → Print → Save as PDF.**
4. **Open the PDF and read it before you upload.** The PDF will look complete even when it isn't. Every heading and prompt prints whether or not the code ran. What matters is the **output**: under each code cell you should see a table, a number, or a plot. A red error box, or `In [ ]` with nothing beneath it, means that part did not run and will be graded as missing. Also check that your memo printed in full and that no plot is cut off at a page break.
5. Upload the PDF to Canvas.